Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration for clearer visualization
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

print("Environment setup complete.")

Environment setup complete.


Data Loading

In [2]:
# UPDATE THESE PATHS to your actual file locations
path_ev = "Dataset 1_EV charging reports.csv"
path_traffic = "Dataset 6_Local traffic distribution.csv"
path_weather = "Dataset 7_Weather Data.csv"

# Load data with correct delimiters
df_ev = pd.read_csv(path_ev, sep=';')
df_traffic = pd.read_csv(path_traffic, sep=';')
df_weather = pd.read_csv(path_weather) # Weather usually uses standard comma

print("Raw Data Loaded.")

Raw Data Loaded.


Data Cleaning

In [3]:
# 1. Parse Dates
# EV: Format Day.Month.Year Hour:Minute
df_ev['Start_plugin'] = pd.to_datetime(df_ev['Start_plugin'], format='%d.%m.%Y %H:%M', errors='coerce')

# Traffic: Format Day.Month.Year Hour:Minute
df_traffic['Date_from'] = pd.to_datetime(df_traffic['Date_from'], format='%d.%m.%Y %H:%M', errors='coerce')

# Weather: Format Year-Month-Day
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])

# 2. Fix Numeric Formats
# Ensure Energy (kWh) is float
if df_ev['El_kWh'].dtype == object:
    df_ev['El_kWh'] = df_ev['El_kWh'].astype(str).str.replace(',', '.').astype(float)

print(f"Date Range EV: {df_ev['Start_plugin'].min()} to {df_ev['Start_plugin'].max()}")

Date Range EV: 2018-12-21 10:20:00 to 2020-01-31 20:42:00


Aggregating

In [4]:
# Create a base "Hour" column to group by
df_ev['date_hour'] = df_ev['Start_plugin'].dt.floor('h')

# Group by Hour and Sum Energy
hourly_demand = df_ev.groupby('date_hour')['El_kWh'].sum().reset_index()
hourly_demand.rename(columns={'El_kWh': 'total_load_kwh', 'date_hour': 'timestamp'}, inplace=True)

# Fill missing hours with 0 (No cars charging = 0 Demand)
full_range = pd.date_range(start=hourly_demand['timestamp'].min(),
                           end=hourly_demand['timestamp'].max(),
                           freq='h')

hourly_demand = hourly_demand.set_index('timestamp').reindex(full_range).fillna(0).reset_index()
hourly_demand.rename(columns={'index': 'timestamp'}, inplace=True)

print(f"Target Created. Total Hours: {len(hourly_demand)}")
hourly_demand.head()

Target Created. Total Hours: 9755


,timestamp,total_load_kwh
0,2018-12-21 10:00:00,1.17
1,2018-12-21 11:00:00,29.87
2,2018-12-21 12:00:00,0.00
3,2018-12-21 13:00:00,0.00
4,2018-12-21 14:00:00,0.00


Merging

In [5]:
# --- PREPARE TRAFFIC ---
# Select timestamp and one representative sensor
traffic_cols = ['Date_from', 'KROPPAN BRU']
df_traffic_clean = df_traffic[traffic_cols].copy()
df_traffic_clean.rename(columns={'Date_from': 'timestamp', 'KROPPAN BRU': 'traffic_volume'}, inplace=True)

# Force numeric
df_traffic_clean['traffic_volume'] = pd.to_numeric(df_traffic_clean['traffic_volume'], errors='coerce').fillna(0)

# Merge Demand + Traffic
df_main = pd.merge(hourly_demand, df_traffic_clean, on='timestamp', how='left')

# --- PREPARE WEATHER ---
# Create a date key for joining
df_main['date_only'] = df_main['timestamp'].dt.normalize()

# Merge Weather
weather_cols = ['datetime', 'temp', 'precip', 'wind_spd']
df_main = pd.merge(df_main, df_weather[weather_cols], left_on='date_only', right_on='datetime', how='left')

# Cleanup
df_main.drop(columns=['date_only', 'datetime'], inplace=True)
df_main.fillna(method='ffill', inplace=True) # Forward fill weather gaps

print("External features merged.")

External features merged.


Feature Engineering

In [6]:
# 1. Basic Time Features
df_main['hour'] = df_main['timestamp'].dt.hour
df_main['day_of_week'] = df_main['timestamp'].dt.dayofweek
df_main['month'] = df_main['timestamp'].dt.month
df_main['is_weekend'] = df_main['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# 2. Cyclical Encoding (Hour & Month)
df_main['hour_sin'] = np.sin(2 * np.pi * df_main['hour'] / 24)
df_main['hour_cos'] = np.cos(2 * np.pi * df_main['hour'] / 24)
df_main['month_sin'] = np.sin(2 * np.pi * df_main['month'] / 12)
df_main['month_cos'] = np.cos(2 * np.pi * df_main['month'] / 12)

# 3. Lag Features (The Memory)
target = 'total_load_kwh'
df_main['lag_1h'] = df_main[target].shift(1)
df_main['lag_24h'] = df_main[target].shift(24) # Same time yesterday
df_main['lag_168h'] = df_main[target].shift(168) # Same time last week
df_main['rolling_mean_24h'] = df_main[target].shift(1).rolling(window=24).mean()

# Drop NaN values created by lags
df_final = df_main.dropna().reset_index(drop=True)

print("Feature Engineering Complete.")

Feature Engineering Complete.


Save Data


In [7]:
# Save to CSV for the Modeling Notebook
df_final.to_csv('processed_ev_data.csv', index=False)
print("File saved: 'processed_ev_data.csv'")

File saved: 'processed_ev_data.csv'
